# EDA + Anomaly Detection Analysis
## AI Госзакупки РК — Исследование данных

**Цель:** Понять распределение данных, протестировать методы детекции аномалий, выбрать оптимальные пороги.

### Содержание:
1. Подключение и обзор данных
2. EDA: распределения цен, объёмов
3. Статистические методы (IQR, Z-score, MAD)
4. ML-модели (Isolation Forest, LOF)
5. Сравнение методов + метрики
6. Fair Price валидация
7. Выводы и рекомендации

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import psycopg2
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

# --- DB Connection ---
conn = psycopg2.connect(
    host='localhost', port=5433,
    dbname='goszakup', user='goszakup', password='gos123'
)
print('Connected to PostgreSQL')

## 1. Обзор данных

In [ ]:
# Сколько данных у нас есть?
overview = pd.read_sql("""
    SELECT 'subjects' AS entity, COUNT(*) AS cnt FROM subjects
    UNION ALL SELECT 'announcements', COUNT(*) FROM announcements
    UNION ALL SELECT 'lots', COUNT(*) FROM lots
    UNION ALL SELECT 'contracts', COUNT(*) FROM contracts
    UNION ALL SELECT 'contract_subjects', COUNT(*) FROM contract_subjects
    UNION ALL SELECT 'plans', COUNT(*) FROM plans
    ORDER BY cnt DESC
""", conn)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(overview['entity'], overview['cnt'], color='steelblue')
ax.set_xlabel('Количество записей')
ax.set_title('Объём данных по таблицам')
for bar, val in zip(bars, overview['cnt']):
    ax.text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2, 
            f'{val:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

overview

In [ ]:
# Качество данных: NULL-полноты
quality = pd.read_sql("""
    SELECT 
        'contract_subjects' AS table_name,
        COUNT(*) AS total,
        COUNT(enstru_code) AS has_enstru,
        COUNT(price_per_unit) AS has_price,
        COUNT(quantity) AS has_quantity,
        COUNT(delivery_kato) AS has_kato,
        ROUND(100.0 * COUNT(enstru_code) / COUNT(*), 1) AS pct_enstru,
        ROUND(100.0 * COUNT(price_per_unit) / COUNT(*), 1) AS pct_price,
        ROUND(100.0 * COUNT(quantity) / COUNT(*), 1) AS pct_quantity
    FROM contract_subjects
""", conn)

print('=== Полнота данных contract_subjects ===')
print(f'Всего записей:        {quality["total"].iloc[0]:>10,.0f}')
print(f'С ENSTRU кодом:       {quality["has_enstru"].iloc[0]:>10,.0f}  ({quality["pct_enstru"].iloc[0]}%)')
print(f'С ценой за единицу:   {quality["has_price"].iloc[0]:>10,.0f}  ({quality["pct_price"].iloc[0]}%)')
print(f'С количеством:        {quality["has_quantity"].iloc[0]:>10,.0f}  ({quality["pct_quantity"].iloc[0]}%)')

## 2. EDA: Распределение цен

In [ ]:
# Загрузим данные по ценам из contract_subjects
prices_df = pd.read_sql("""
    SELECT 
        cs.id, cs.enstru_code, cs.price_per_unit, cs.quantity, cs.total_price,
        cs.delivery_kato, rk.region AS delivery_region,
        c.sign_date, c.customer_bin, c.supplier_bin, c.contract_sum
    FROM contract_subjects cs
    JOIN contracts c ON cs.contract_id = c.id
    LEFT JOIN ref_kato rk ON cs.delivery_kato = rk.code
    WHERE cs.price_per_unit IS NOT NULL 
      AND cs.price_per_unit > 0
      AND cs.enstru_code IS NOT NULL
""", conn)

print(f'Записей с ценами: {len(prices_df):,}')
print(f'Уникальных ENSTRU: {prices_df["enstru_code"].nunique():,}')
print(f'Уникальных регионов: {prices_df["delivery_region"].nunique()}')
print(f'\nОписательная статистика price_per_unit:')
prices_df['price_per_unit'].describe()

In [ ]:
# Распределение цен (лог-шкала, т.к. цены сильно варьируются)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Гистограмма цен (log)
log_prices = np.log10(prices_df['price_per_unit'].clip(lower=0.01))
axes[0].hist(log_prices, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('log10(price_per_unit)')
axes[0].set_ylabel('Частота')
axes[0].set_title('Распределение цен (log10)')
axes[0].axvline(log_prices.median(), color='red', linestyle='--', label=f'Медиана: {10**log_prices.median():,.0f}')
axes[0].legend()

# 2. Box plot цен по топ-10 регионам
top_regions = prices_df['delivery_region'].value_counts().head(10).index
regional = prices_df[prices_df['delivery_region'].isin(top_regions)].copy()
regional['log_price'] = np.log10(regional['price_per_unit'].clip(lower=0.01))
sns.boxplot(data=regional, x='delivery_region', y='log_price', ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('log10(price_per_unit)')
axes[1].set_title('Цены по регионам (топ-10)')

# 3. Количество записей по годам
prices_df['year'] = pd.to_datetime(prices_df['sign_date']).dt.year
year_counts = prices_df['year'].value_counts().sort_index()
axes[2].bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
axes[2].set_xlabel('Год')
axes[2].set_ylabel('Количество записей')
axes[2].set_title('Записи по годам')
for i, v in enumerate(year_counts.values):
    axes[2].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Топ-20 ENSTRU кодов по количеству записей (на них будем тестировать)
top_enstru = prices_df.groupby('enstru_code').agg(
    count=('price_per_unit', 'size'),
    median_price=('price_per_unit', 'median'),
    mean_price=('price_per_unit', 'mean'),
    std_price=('price_per_unit', 'std'),
    min_price=('price_per_unit', 'min'),
    max_price=('price_per_unit', 'max'),
).sort_values('count', ascending=False).head(20)

top_enstru['cv'] = top_enstru['std_price'] / top_enstru['mean_price']  # coefficient of variation
top_enstru['range_ratio'] = top_enstru['max_price'] / top_enstru['min_price'].clip(lower=0.01)

print('Топ-20 ENSTRU кодов по количеству записей:')
print('CV = коэфф. вариации (>1 = очень разбросанные цены)')
print('Range = max/min (>100 = подозрительно)')
top_enstru[['count', 'median_price', 'cv', 'range_ratio']].round(2)

## 3. Статистические методы детекции аномалий

In [ ]:
def detect_anomalies_statistical(group_df, method='iqr', threshold=1.5):
    """
    Детектирует аномалии внутри одной группы ENSTRU.
    
    Методы:
    - iqr: Q1 - k*IQR ... Q3 + k*IQR  (k=threshold)
    - zscore: |z| > threshold
    - mad: |X - median| / MAD > threshold
    """
    prices = group_df['price_per_unit'].values
    
    if method == 'iqr':
        q1, q3 = np.percentile(prices, [25, 75])
        iqr = q3 - q1
        if iqr <= 0:
            return pd.Series(False, index=group_df.index)
        lower = q1 - threshold * iqr
        upper = q3 + threshold * iqr
        return (prices < lower) | (prices > upper)
    
    elif method == 'zscore':
        if prices.std() == 0:
            return pd.Series(False, index=group_df.index)
        z = np.abs(stats.zscore(prices))
        return z > threshold
    
    elif method == 'mad':
        median = np.median(prices)
        mad = np.median(np.abs(prices - median))
        if mad == 0:
            return pd.Series(False, index=group_df.index)
        modified_z = 0.6745 * (prices - median) / mad
        return np.abs(modified_z) > threshold


# Применяем все 3 метода к данным с достаточной выборкой (>=10)
enstru_counts = prices_df['enstru_code'].value_counts()
valid_enstru = enstru_counts[enstru_counts >= 10].index
analysis_df = prices_df[prices_df['enstru_code'].isin(valid_enstru)].copy()

print(f'ENSTRU кодов с >= 10 записями: {len(valid_enstru):,}')
print(f'Записей для анализа: {len(analysis_df):,} из {len(prices_df):,}')

# IQR
analysis_df['is_anomaly_iqr'] = analysis_df.groupby('enstru_code', group_keys=False).apply(
    lambda g: pd.Series(detect_anomalies_statistical(g, 'iqr', 1.5), index=g.index)
)

# Z-score
analysis_df['is_anomaly_zscore'] = analysis_df.groupby('enstru_code', group_keys=False).apply(
    lambda g: pd.Series(detect_anomalies_statistical(g, 'zscore', 3.0), index=g.index)
)

# MAD
analysis_df['is_anomaly_mad'] = analysis_df.groupby('enstru_code', group_keys=False).apply(
    lambda g: pd.Series(detect_anomalies_statistical(g, 'mad', 3.5), index=g.index)
)

# Результаты
print(f'\n=== Аномалии по методам ===')
print(f'IQR (k=1.5):      {analysis_df["is_anomaly_iqr"].sum():>6,} ({100*analysis_df["is_anomaly_iqr"].mean():.2f}%)')
print(f'Z-score (z>3):     {analysis_df["is_anomaly_zscore"].sum():>6,} ({100*analysis_df["is_anomaly_zscore"].mean():.2f}%)')
print(f'MAD (z>3.5):       {analysis_df["is_anomaly_mad"].sum():>6,} ({100*analysis_df["is_anomaly_mad"].mean():.2f}%)')

In [ ]:
# Пересечение методов (consensus = >1 метод считает аномалией)
analysis_df['anomaly_score'] = (
    analysis_df['is_anomaly_iqr'].astype(int) + 
    analysis_df['is_anomaly_zscore'].astype(int) + 
    analysis_df['is_anomaly_mad'].astype(int)
)

# Venn-подобная статистика
print('=== Пересечение методов (consensus) ===')
print(f'Ни один метод:   {(analysis_df["anomaly_score"]==0).sum():>6,}')
print(f'1 метод:         {(analysis_df["anomaly_score"]==1).sum():>6,}')
print(f'2 метода:        {(analysis_df["anomaly_score"]==2).sum():>6,}')
print(f'Все 3 метода:    {(analysis_df["anomaly_score"]==3).sum():>6,}  <- высокая уверенность')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
score_counts = analysis_df['anomaly_score'].value_counts().sort_index()
labels = ['Норма', '1 метод', '2 метода', 'Все 3 метода']
colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[0].pie(score_counts.values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Распределение аномалий по consensus')

# Top anomalies by deviation
confirmed_anomalies = analysis_df[analysis_df['anomaly_score'] >= 2].copy()
if not confirmed_anomalies.empty:
    # Считаем отклонение от медианы группы
    group_medians = analysis_df.groupby('enstru_code')['price_per_unit'].median()
    confirmed_anomalies['median_price'] = confirmed_anomalies['enstru_code'].map(group_medians)
    confirmed_anomalies['deviation_pct'] = (
        (confirmed_anomalies['price_per_unit'] - confirmed_anomalies['median_price']) 
        / confirmed_anomalies['median_price'] * 100
    )
    
    # Гистограмма отклонений
    dev = confirmed_anomalies['deviation_pct'].clip(-500, 500)
    axes[1].hist(dev, bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='black', linestyle='-', linewidth=1)
    axes[1].set_xlabel('Отклонение от медианы (%)')
    axes[1].set_ylabel('Количество')
    axes[1].set_title(f'Подтверждённые аномалии (2+ метода): {len(confirmed_anomalies):,}')

plt.tight_layout()
plt.show()

## 4. ML-модели: Isolation Forest + LOF

In [ ]:
# Готовим features для ML
# Для каждой записи: log(price), отклонение от медианы группы, z-score внутри группы

def build_features(df):
    """Строит feature-матрицу для ML-моделей."""
    feat = df.copy()
    
    # Медиана и стд по ENSTRU группе
    group_stats = df.groupby('enstru_code')['price_per_unit'].agg(['median', 'std', 'mean']).reset_index()
    group_stats.columns = ['enstru_code', 'group_median', 'group_std', 'group_mean']
    feat = feat.merge(group_stats, on='enstru_code', how='left')
    
    # Features
    feat['log_price'] = np.log1p(feat['price_per_unit'])
    feat['deviation_from_median'] = (feat['price_per_unit'] - feat['group_median']) / feat['group_median'].clip(lower=0.01)
    feat['zscore_in_group'] = (feat['price_per_unit'] - feat['group_mean']) / feat['group_std'].clip(lower=0.01)
    feat['log_quantity'] = np.log1p(feat['quantity'].fillna(1))
    feat['price_x_qty'] = feat['price_per_unit'] * feat['quantity'].fillna(1)
    feat['log_total'] = np.log1p(feat['price_x_qty'])
    
    feature_cols = ['log_price', 'deviation_from_median', 'zscore_in_group', 'log_quantity', 'log_total']
    
    # Убираем NaN и Inf
    for col in feature_cols:
        feat[col] = feat[col].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    return feat, feature_cols


ml_df, feature_cols = build_features(analysis_df)
X = ml_df[feature_cols].values

# Нормализация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix: {X_scaled.shape}')
print(f'Features: {feature_cols}')
print(f'\nFeature statistics (scaled):')
pd.DataFrame(X_scaled, columns=feature_cols).describe().round(2)

In [ ]:
# === Isolation Forest ===
# contamination = ожидаемая доля аномалий (начинаем с 5%)
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)
iso_labels = iso_forest.fit_predict(X_scaled)  # 1 = normal, -1 = anomaly
iso_scores = iso_forest.decision_function(X_scaled)  # чем ниже, тем больше аномалия

ml_df['iso_forest_label'] = (iso_labels == -1)  # True = anomaly
ml_df['iso_forest_score'] = iso_scores

print(f'=== Isolation Forest ===')
print(f'Аномалий: {(iso_labels == -1).sum():,} ({100*(iso_labels == -1).mean():.2f}%)')
print(f'Нормальных: {(iso_labels == 1).sum():,}')
print(f'\nScore distribution:')
print(f'  Anomalies  mean score: {iso_scores[iso_labels == -1].mean():.4f}')
print(f'  Normal     mean score: {iso_scores[iso_labels == 1].mean():.4f}')

In [ ]:
# === Local Outlier Factor ===
# LOF работает локально — хорошо для кластерных данных
# Используем подвыборку если данных слишком много (LOF = O(n^2))
MAX_LOF_SAMPLES = 50_000

if len(X_scaled) > MAX_LOF_SAMPLES:
    print(f'LOF: подвыборка {MAX_LOF_SAMPLES:,} из {len(X_scaled):,}')
    np.random.seed(42)
    lof_idx = np.random.choice(len(X_scaled), MAX_LOF_SAMPLES, replace=False)
    X_lof = X_scaled[lof_idx]
else:
    lof_idx = np.arange(len(X_scaled))
    X_lof = X_scaled

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    n_jobs=-1,
)
lof_labels = lof.fit_predict(X_lof)  # 1 = normal, -1 = anomaly
lof_scores = lof.negative_outlier_factor_

# Записываем результаты (только для подвыборки)
ml_df['lof_label'] = np.nan
ml_df.iloc[lof_idx, ml_df.columns.get_loc('lof_label')] = (lof_labels == -1)

print(f'\n=== Local Outlier Factor ===')
print(f'Аномалий: {(lof_labels == -1).sum():,} ({100*(lof_labels == -1).mean():.2f}%)')
print(f'\nLOF Score distribution:')
print(f'  Anomalies  mean LOF: {lof_scores[lof_labels == -1].mean():.4f}')
print(f'  Normal     mean LOF: {lof_scores[lof_labels == 1].mean():.4f}')

In [ ]:
# Визуализация Isolation Forest scores
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution of anomaly scores
axes[0].hist(iso_scores[iso_labels == 1], bins=50, alpha=0.7, label='Normal', color='steelblue')
axes[0].hist(iso_scores[iso_labels == -1], bins=50, alpha=0.7, label='Anomaly', color='#e74c3c')
axes[0].set_xlabel('Isolation Forest Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Isolation Forest: Score Distribution')
axes[0].legend()

# 2. Scatter: price deviation vs anomaly score
scatter_idx = np.random.choice(len(ml_df), min(5000, len(ml_df)), replace=False)
scatter_data = ml_df.iloc[scatter_idx]
colors_scatter = ['#e74c3c' if a else '#2ecc71' for a in scatter_data['iso_forest_label']]
axes[1].scatter(
    scatter_data['deviation_from_median'].clip(-5, 5),
    scatter_data['iso_forest_score'],
    c=colors_scatter, alpha=0.3, s=5
)
axes[1].set_xlabel('Отклонение от медианы (ratio)')
axes[1].set_ylabel('IF Score')
axes[1].set_title('Deviation vs IF Score')
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)

# 3. Feature importance (по средней разнице нормальных vs аномалий)
importance = []
for col in feature_cols:
    normal_mean = ml_df.loc[~ml_df['iso_forest_label'], col].mean()
    anomaly_mean = ml_df.loc[ml_df['iso_forest_label'], col].mean()
    importance.append(abs(anomaly_mean - normal_mean))

axes[2].barh(feature_cols, importance, color='steelblue')
axes[2].set_xlabel('|mean_anomaly - mean_normal|')
axes[2].set_title('Feature Importance (IF)')

plt.tight_layout()
plt.show()

## 5. Сравнение методов + Метрики

In [ ]:
# Сравнение всех методов
# Без ground truth используем consensus (agreement) как proxy

methods = {
    'IQR (k=1.5)': ml_df['is_anomaly_iqr'],
    'Z-score (z>3)': ml_df['is_anomaly_zscore'],
    'MAD (z>3.5)': ml_df['is_anomaly_mad'],
    'Isolation Forest': ml_df['iso_forest_label'],
}

print('=== Сравнение методов ===')
print(f'{"Метод":<25} {"Аномалий":>10} {"% от всех":>10}')
print('-' * 50)
for name, labels in methods.items():
    n_anom = labels.sum()
    pct = 100 * labels.mean()
    print(f'{name:<25} {n_anom:>10,.0f} {pct:>9.2f}%')

# Матрица согласия (agreement matrix)
print('\n=== Матрица согласия (% пересечения) ===')
method_names = list(methods.keys())
agreement = pd.DataFrame(index=method_names, columns=method_names, dtype=float)

for i, (n1, l1) in enumerate(methods.items()):
    for j, (n2, l2) in enumerate(methods.items()):
        both = (l1 & l2).sum()
        total = l1.sum()
        agreement.loc[n1, n2] = round(100 * both / max(total, 1), 1)

agreement

In [ ]:
# Визуализация согласия методов (heatmap)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
sns.heatmap(agreement.astype(float), annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('Согласие методов (% пересечения аномалий)')

# Consensus histogram
ml_df['ml_consensus'] = (
    ml_df['is_anomaly_iqr'].astype(int) + 
    ml_df['is_anomaly_zscore'].astype(int) + 
    ml_df['is_anomaly_mad'].astype(int) +
    ml_df['iso_forest_label'].astype(int)
)

consensus_counts = ml_df['ml_consensus'].value_counts().sort_index()
colors_bar = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#c0392b']
axes[1].bar(consensus_counts.index, consensus_counts.values, color=colors_bar[:len(consensus_counts)])
axes[1].set_xlabel('Количество методов, нашедших аномалию')
axes[1].set_ylabel('Количество записей')
axes[1].set_title('Consensus: сколько методов согласны')
for i, v in enumerate(consensus_counts.values):
    axes[1].text(consensus_counts.index[i], v + 100, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Итоговая метрика
high_confidence = (ml_df['ml_consensus'] >= 3).sum()
print(f'\nВысокая уверенность (3+ метода): {high_confidence:,} аномалий')
print(f'Это {100 * high_confidence / len(ml_df):.2f}% от всех записей')

## 6. Детальный анализ найденных аномалий

In [ ]:
# Топ-20 самых подозрительных записей (consensus >= 3)
high_anomalies = ml_df[ml_df['ml_consensus'] >= 3].copy()

if not high_anomalies.empty:
    # Считаем отклонение
    group_medians = ml_df.groupby('enstru_code')['price_per_unit'].median()
    high_anomalies['group_median'] = high_anomalies['enstru_code'].map(group_medians)
    high_anomalies['deviation_pct'] = (
        (high_anomalies['price_per_unit'] - high_anomalies['group_median']) 
        / high_anomalies['group_median'] * 100
    )
    
    top_anomalies = high_anomalies.nlargest(20, 'deviation_pct')
    
    print('=== Топ-20 завышенных цен (consensus 3+ метода) ===')
    display_cols = ['id', 'enstru_code', 'price_per_unit', 'group_median', 
                    'deviation_pct', 'customer_bin', 'supplier_bin', 'ml_consensus']
    print(top_anomalies[display_cols].to_string(index=False))
else:
    print('Аномалий с высокой уверенностью не найдено')

In [ ]:
# Визуализация конкретного ENSTRU с аномалиями
if not high_anomalies.empty:
    # Берём ENSTRU с наибольшим количеством аномалий
    target_enstru = high_anomalies['enstru_code'].value_counts().index[0]
    enstru_data = ml_df[ml_df['enstru_code'] == target_enstru].copy()
    enstru_anomalies = enstru_data[enstru_data['ml_consensus'] >= 3]
    enstru_normal = enstru_data[enstru_data['ml_consensus'] < 3]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Box plot
    axes[0].boxplot(enstru_data['price_per_unit'], vert=True)
    if not enstru_anomalies.empty:
        axes[0].scatter(
            [1] * len(enstru_anomalies), enstru_anomalies['price_per_unit'],
            color='red', zorder=5, s=50, label=f'Аномалии ({len(enstru_anomalies)})', marker='x'
        )
    axes[0].set_title(f'ENSTRU: {target_enstru}\n(всего {len(enstru_data)} записей)')
    axes[0].set_ylabel('Цена за единицу')
    axes[0].legend()
    
    # Scatter по времени
    if enstru_data['sign_date'].notna().any():
        axes[1].scatter(
            pd.to_datetime(enstru_normal['sign_date']), enstru_normal['price_per_unit'],
            alpha=0.5, s=20, color='steelblue', label='Нормальные'
        )
        if not enstru_anomalies.empty:
            axes[1].scatter(
                pd.to_datetime(enstru_anomalies['sign_date']), enstru_anomalies['price_per_unit'],
                color='red', s=60, marker='x', label='Аномалии', zorder=5
            )
        axes[1].set_xlabel('Дата подписания')
        axes[1].set_ylabel('Цена за единицу')
        axes[1].set_title('Цены во времени')
        axes[1].legend()
        plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()

## 7. Fair Price валидация

In [ ]:
# Проверяем формулу Fair Price на реальных данных
fair_price_data = pd.read_sql("""
    SELECT 
        enstru_code,
        SUM(sample_size) AS total_samples,
        SUM(median_price * sample_size) / NULLIF(SUM(sample_size), 0) AS weighted_median,
        MIN(min_price) AS min_price,
        MAX(max_price) AS max_price,
        SUM(q1 * sample_size) / NULLIF(SUM(sample_size), 0) AS weighted_q1,
        SUM(q3 * sample_size) / NULLIF(SUM(sample_size), 0) AS weighted_q3
    FROM mv_price_statistics
    WHERE enstru_code IS NOT NULL AND sample_size >= 5
    GROUP BY enstru_code
    HAVING SUM(sample_size) >= 10
    ORDER BY SUM(sample_size) DESC
    LIMIT 50
""", conn)

print(f'ENSTRU кодов с достаточной выборкой: {len(fair_price_data)}')
print(f'\nТоп-15 по количеству данных:')

fp_display = fair_price_data.head(15).copy()
fp_display['iqr'] = fp_display['weighted_q3'] - fp_display['weighted_q1']
fp_display['cv'] = fp_display['iqr'] / fp_display['weighted_median'].clip(lower=0.01)
fp_display['range'] = fp_display['max_price'] / fp_display['min_price'].clip(lower=0.01)

fp_display[['enstru_code', 'total_samples', 'weighted_median', 'weighted_q1', 
            'weighted_q3', 'cv', 'range']].round(2)

In [ ]:
# Confidence level distribution
confidence_data = pd.read_sql("""
    SELECT 
        enstru_code,
        SUM(sample_size) AS total_samples
    FROM mv_price_statistics
    WHERE enstru_code IS NOT NULL
    GROUP BY enstru_code
""", conn)

confidence_data['confidence'] = confidence_data['total_samples'].apply(
    lambda x: 'high' if x >= 30 else 'medium' if x >= 10 else 'low'
)

conf_counts = confidence_data['confidence'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_conf = {'high': '#2ecc71', 'medium': '#f1c40f', 'low': '#e74c3c'}
axes[0].bar(conf_counts.index, conf_counts.values, 
            color=[colors_conf.get(c, 'gray') for c in conf_counts.index])
axes[0].set_xlabel('Уровень доверия')
axes[0].set_ylabel('Количество ENSTRU кодов')
axes[0].set_title('Fair Price: распределение по уровню доверия')
for i, v in enumerate(conf_counts.values):
    axes[0].text(i, v + 10, f'{v:,}', ha='center')

# Sample size distribution
axes[1].hist(np.log10(confidence_data['total_samples'].clip(lower=1)), bins=50, color='steelblue')
axes[1].set_xlabel('log10(sample_size)')
axes[1].set_ylabel('Количество ENSTRU')
axes[1].set_title('Распределение размера выборки')
axes[1].axvline(np.log10(10), color='orange', linestyle='--', label='medium (10)')
axes[1].axvline(np.log10(30), color='green', linestyle='--', label='high (30)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nFair Price confidence:')
print(f'  High (>=30 samples):   {conf_counts.get("high", 0):>6,} ENSTRU кодов')
print(f'  Medium (10-29):        {conf_counts.get("medium", 0):>6,} ENSTRU кодов')
print(f'  Low (<10):             {conf_counts.get("low", 0):>6,} ENSTRU кодов')

## 8. Supplier Concentration Analysis

In [ ]:
# Концентрация поставщиков
concentration = pd.read_sql("""
    WITH customer_totals AS (
        SELECT customer_bin, SUM(contract_sum) AS total
        FROM contracts WHERE contract_sum > 0
        GROUP BY customer_bin
    ),
    supplier_shares AS (
        SELECT c.customer_bin, c.supplier_bin,
               SUM(c.contract_sum) AS supplier_total,
               ct.total AS customer_total,
               100.0 * SUM(c.contract_sum) / NULLIF(ct.total, 0) AS share_pct
        FROM contracts c
        JOIN customer_totals ct ON c.customer_bin = ct.customer_bin
        WHERE c.contract_sum > 0
        GROUP BY c.customer_bin, c.supplier_bin, ct.total
    )
    SELECT ss.customer_bin, s1.name_ru AS customer_name,
           ss.supplier_bin, s2.name_ru AS supplier_name,
           ss.supplier_total, ss.customer_total, ss.share_pct
    FROM supplier_shares ss
    LEFT JOIN subjects s1 ON ss.customer_bin = s1.bin
    LEFT JOIN subjects s2 ON ss.supplier_bin = s2.bin
    WHERE ss.share_pct >= 50
    ORDER BY ss.share_pct DESC
""", conn)

print(f'Пар заказчик-поставщик с долей >= 50%: {len(concentration)}')

if not concentration.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Распределение долей
    axes[0].hist(concentration['share_pct'], bins=20, color='#e74c3c', edgecolor='white')
    axes[0].set_xlabel('Доля поставщика (%)')
    axes[0].set_ylabel('Количество пар')
    axes[0].set_title('Концентрация поставщиков (>50%)')
    axes[0].axvline(80, color='black', linestyle='--', label='Порог 80%')
    axes[0].legend()
    
    # Топ по сумме
    top_conc = concentration.head(10).copy()
    top_conc['label'] = top_conc['customer_name'].str[:30] + '\n→ ' + top_conc['supplier_name'].str[:30]
    axes[1].barh(range(len(top_conc)), top_conc['share_pct'], color='#e74c3c')
    axes[1].set_yticks(range(len(top_conc)))
    axes[1].set_yticklabels(top_conc['label'], fontsize=7)
    axes[1].set_xlabel('Доля (%)')
    axes[1].set_title('Топ-10 по концентрации')
    
    plt.tight_layout()
    plt.show()

    print('\nТоп-10 подозрительных пар:')
    print(concentration[['customer_name', 'supplier_name', 'share_pct', 'supplier_total']].head(10).to_string(index=False))

## 9. Итоговые метрики системы

In [ ]:
# === ИТОГОВЫЙ ОТЧЁТ ===
print('=' * 60)
print('  ИТОГОВЫЙ ОТЧЁТ: AI Госзакупки — Anomaly Detection')
print('=' * 60)

total_records = len(prices_df)
analyzed_records = len(analysis_df)

print(f'\n1. ДАННЫЕ')
print(f'   Всего записей с ценами:     {total_records:>10,}')
print(f'   Записей для анализа (>=10):  {analyzed_records:>10,}')
print(f'   Уникальных ENSTRU:           {prices_df["enstru_code"].nunique():>10,}')
print(f'   Анализируемых ENSTRU:        {len(valid_enstru):>10,}')

print(f'\n2. АНОМАЛИИ ПО МЕТОДАМ')
for name, labels in methods.items():
    n = labels.sum()
    print(f'   {name:<25}: {n:>8,}  ({100*labels.mean():.2f}%)')

consensus_3 = (ml_df['ml_consensus'] >= 3).sum()
consensus_4 = (ml_df['ml_consensus'] >= 4).sum()
print(f'\n3. CONSENSUS (ДОВЕРИЕ)')
print(f'   3+ метода (высокое):    {consensus_3:>8,}  ({100*consensus_3/analyzed_records:.2f}%)')
print(f'   Все 4 метода (оч.выс.): {consensus_4:>8,}  ({100*consensus_4/analyzed_records:.2f}%)')

print(f'\n4. FAIR PRICE')
print(f'   ENSTRU с high confidence:   {conf_counts.get("high", 0):>6,}')
print(f'   ENSTRU с medium confidence: {conf_counts.get("medium", 0):>6,}')
print(f'   ENSTRU с low confidence:    {conf_counts.get("low", 0):>6,}')

print(f'\n5. SUPPLIER CONCENTRATION')
print(f'   Пар с долей >= 80%:  {len(concentration[concentration["share_pct"] >= 80]):>6,}')
print(f'   Пар с долей >= 90%:  {len(concentration[concentration["share_pct"] >= 90]):>6,}')
print(f'   Пар с долей >= 95%:  {len(concentration[concentration["share_pct"] >= 95]):>6,}')

print(f'\n6. РЕКОМЕНДАЦИИ')
print(f'   - Использовать consensus (3+ метода) как основной критерий аномалии')
print(f'   - Isolation Forest хорошо дополняет статистику, ловит многомерные аномалии')
print(f'   - Fair Price надёжен только для ENSTRU с high confidence (>=30 samples)')
print(f'   - Supplier concentration >= 80% — важный индикатор рисков')
print('=' * 60)

In [ ]:
# Закрываем соединение
conn.close()
print('Done!')